In [4]:
!pip install pyreadstat

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.0 MB 656.4 kB/s eta 0:00:04
   ------------ --------------------------- 0.7/2.0 MB 5.2 MB/s eta 0:00:01
   ---------------- ----------------------- 0.9/2.0 MB 4.9 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 4.7 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 4.7 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 4.7 MB/s eta 0:00:01
   --------------------- ------------------ 1.1/2.0 MB 2.9 MB/s eta 0:00:01
   -------------------------------- ------- 1.7/2.0 MB 3.9 MB/s eta 0:00:01
   ------------------------------------ --- 1.9/2.0 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------  2.0/2.0 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------  2.0/2.0 MB 3.5 MB/s eta 0:00:01
   ---------------------

In [6]:
import pyreadstat
import pandas as pd

cols = ['v005', 'v012', 'v025', 'v106', 'v131', 'v190', 'm15_1']
df, meta = pyreadstat.read_dta('IAIR7EFL.DTA', usecols=cols)

print("Loaded shape:", df.shape)
df.head()

Loaded shape: (724115, 7)


,v005,v012,v025,v106,v131,v190,m15_1
0,193444,22,2,3,991,3,NaN
1,193444,19,2,2,991,3,NaN
2,193444,38,2,1,991,3,NaN
3,193444,40,2,0,991,3,NaN
4,193444,22,2,3,991,3,NaN


In [8]:
# Reload with 's116' (Caste/Tribe category) instead of 'v131'
cols = ['v005', 'v012', 'v025', 'v106', 's116', 'v190', 'm15_1']
df, meta = pyreadstat.read_dta('IAIR7EFL.DTA', usecols=cols)

# 1. Listwise deletion for rows missing the target variable (no recent birth)
df_clean = df.dropna(subset=['m15_1']).copy()

# 2. Construct binary target: 1 = Institutional Delivery (codes 20-33), 0 = Home/Other (codes 10-13, 96)
# Exclude '99' (Missing)
df_clean = df_clean[df_clean['m15_1'] != 99]
df_clean['institutional_delivery'] = df_clean['m15_1'].apply(lambda x: 1 if 20 <= x <= 33 else 0)

print("Cleaned dataset shape (women with recent births):", df_clean.shape)
print("\nTarget Variable Distribution (Class Imbalance):")
print(df_clean['institutional_delivery'].value_counts(normalize=True) * 100)

Cleaned dataset shape (women with recent births): (176843, 8)

Target Variable Distribution (Class Imbalance):
institutional_delivery
1    87.779556
0    12.220444
Name: proportion, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pandas as pd

# 1. Separate Predictors (X) and Target (y)
# (Dropping the original target 'm15_1' and the sample weight 'v005' from the predictors)
X = df_clean[['v012', 'v025', 'v106', 's116', 'v190']].copy()
y = df_clean['institutional_delivery']

# 2. Handle Missing Values
# Median for numerical (Age: v012), Mode for categorical
X['v012'] = X['v012'].fillna(X['v012'].median())

for col in ['v025', 'v106', 's116', 'v190']:
    X[col] = X[col].fillna(X[col].mode()[0])

# 3. Encoding
# Wealth (v190) and Education (v106) are already ordinal numbers in the dataset.
# Residence (v025) is currently 1=Urban, 2=Rural. Let's convert to 0=Urban, 1=Rural binary.
X['v025'] = X['v025'].replace({1: 0, 2: 1})

# Caste/Tribe (s116) is nominal, so we apply One-Hot Encoding
X = pd.get_dummies(X, columns=['s116'], drop_first=True)

# 4. Train-Test Split (80% Train, 20% Test)
# We use 'stratify=y' to ensure the 88/12 split is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Apply SMOTE to the Training Data ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Original Training Class Distribution:\n", y_train.value_counts())
print("\nSMOTE Training Class Distribution:\n", y_train_smote.value_counts())
print("\nTest Set Size (Untouched):", X_test.shape[0])

C:\Users\Hp\AppData\Local\Temp\ipykernel_11664\1039005943.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].fillna(X[col].mode()[0])


Original Training Class Distribution:
 institutional_delivery
1    124185
0     17289
Name: count, dtype: int64

SMOTE Training Class Distribution:
 institutional_delivery
1    124185
0    124185
Name: count, dtype: int64

Test Set Size (Untouched): 35369


In [10]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pandas as pd

# 1. Separate Predictors (X) and Target (y)
# (Dropping the original target 'm15_1' and the sample weight 'v005' from the predictors)
X = df_clean[['v012', 'v025', 'v106', 's116', 'v190']].copy()
y = df_clean['institutional_delivery']

# 2. Handle Missing Values
# Median for numerical (Age: v012), Mode for categorical
X['v012'] = X['v012'].fillna(X['v012'].median())

for col in ['v025', 'v106', 's116', 'v190']:
    X[col] = X[col].fillna(X[col].mode()[0])

# 3. Encoding
# Wealth (v190) and Education (v106) are already ordinal numbers in the dataset.
# Residence (v025) is currently 1=Urban, 2=Rural. Let's convert to 0=Urban, 1=Rural binary.
X['v025'] = X['v025'].replace({1: 0, 2: 1})

# Caste/Tribe (s116) is nominal, so we apply One-Hot Encoding
X = pd.get_dummies(X, columns=['s116'], drop_first=True)

# 4. Train-Test Split (80% Train, 20% Test)
# We use 'stratify=y' to ensure the 88/12 split is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Apply SMOTE to the Training Data ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Original Training Class Distribution:\n", y_train.value_counts())
print("\nSMOTE Training Class Distribution:\n", y_train_smote.value_counts())
print("\nTest Set Size (Untouched):", X_test.shape[0])

C:\Users\Hp\AppData\Local\Temp\ipykernel_11664\1039005943.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].fillna(X[col].mode()[0])


Original Training Class Distribution:
 institutional_delivery
1    124185
0     17289
Name: count, dtype: int64

SMOTE Training Class Distribution:
 institutional_delivery
1    124185
0    124185
Name: count, dtype: int64

Test Set Size (Untouched): 35369


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ---------------------------------------------------------
# 1. Train and Evaluate LOGISTIC REGRESSION (Baseline)
# ---------------------------------------------------------
print("--- LOGISTIC REGRESSION ---")
# Train on the SMOTE balanced data
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_smote, y_train_smote)

# Predict on the untouched test data
y_pred_log = log_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print(classification_report(y_test, y_pred_log))

# ---------------------------------------------------------
# 2. Train and Evaluate RANDOM FOREST
# ---------------------------------------------------------
print("\n--- RANDOM FOREST ---")
# To save time and memory on personal laptops, we limit the depth
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_smote, y_train_smote)

# Predict on the untouched test data
y_pred_rf = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf))

--- LOGISTIC REGRESSION ---
Accuracy: 0.6701
              precision    recall  f1-score   support

           0       0.23      0.73      0.35      4322
           1       0.95      0.66      0.78     31047

    accuracy                           0.67     35369
   macro avg       0.59      0.70      0.57     35369
weighted avg       0.86      0.67      0.73     35369


--- RANDOM FOREST ---
Accuracy: 0.6728
              precision    recall  f1-score   support

           0       0.23      0.72      0.35      4322
           1       0.94      0.67      0.78     31047

    accuracy                           0.67     35369
   macro avg       0.59      0.69      0.57     35369
weighted avg       0.86      0.67      0.73     35369



In [16]:
!pip install fairlearn

   ---------------------------------------- 0.0/141.4 kB ? eta -:--:--
   -- ------------------------------------- 10.2/141.4 kB ? eta -:--:--
   ------------------------------------- -- 133.1/141.4 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 141.4/141.4 kB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.6/8.3 MB 18.8 MB/s eta 0:00:01
   --- ------------------------------------ 0.8/8.3 MB 9.8 MB/s eta 0:00:01
   ---- ----------------------------------- 1.0/8.3 MB 6.8 MB/s eta 0:00:02
   ---- ----------------------------------- 1.0/8.3 MB 7.0 MB/s eta 0:00:02
   ---- ----------------------------------- 1.0/8.3 MB 7.0 MB/s eta 0:00:02
   ------- -------------------------------- 1.5/8.3 MB 5.4 MB/s eta 0:00:02
   -------- ------------------------------- 1.7/8.3 MB 5.4 MB/s eta 0:00:02
   --------- ------------------------------ 1.9/8.3 MB 5.2 MB/s eta 0:00:02
   ---------- -----

In [24]:
from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference
from sklearn.metrics import accuracy_score, precision_score, recall_score

# 1. Map the Caste feature and safely FILL any unmapped/missing values
caste_map = {1: 'Scheduled Caste', 2: 'Scheduled Tribe', 3: 'OBC', 4: 'General', 8: 'Don\'t Know'}
caste_test = df_clean.loc[X_test.index, 's116'].map(caste_map).fillna('Other/Unknown')

# 2. Create the Fairlearn MetricFrame
metrics = {
    'accuracy': accuracy_score,
    'recall': recall_score,
    'precision': precision_score
}

metric_frame = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_rf,
    sensitive_features=caste_test
)

# 3. Print Subgroup Metrics
print("--- FAIRNESS AUDIT: PERFORMANCE BY CASTE ---")
print(metric_frame.by_group)

# 4. Calculate Disparity Scores
dp_diff = demographic_parity_difference(y_test, y_pred_rf, sensitive_features=caste_test)
eo_diff = equalized_odds_difference(y_test, y_pred_rf, sensitive_features=caste_test)

print("\n--- FAIRNESS DISPARITY METRICS ---")
print(f"Demographic Parity Difference: {dp_diff:.4f}")
print(f"Equalized Odds Difference: {eo_diff:.4f}")

--- FAIRNESS AUDIT: PERFORMANCE BY CASTE ---
                 accuracy    recall  precision
s116                                          
Don't Know       0.793991  0.859903   0.903553
General          0.813838  0.841968   0.949758
OBC              0.729346  0.739698   0.953848
Other/Unknown    0.672790  0.680339   0.929715
Scheduled Caste  0.686448  0.690407   0.938323
Scheduled Tribe  0.437240  0.297987   0.914002

--- FAIRNESS DISPARITY METRICS ---
Demographic Parity Difference: 0.5942
Equalized Odds Difference: 0.6365
